In [6]:
import pandas as pd
import numpy as np
from astropy.io import fits
from astropy import units as u, constants as const

# --- CONFIGURACIÓN DE ARCHIVOS ---
file_csv_raw = 'eu_actualizado_cif2025.csv'
file_nuevos = 'nuevos_planetas.csv'
file_fits_cif = 'eu_cif2025corr.fit' 
file_correcciones = 'correcciones_manuales.csv'
file_output = 'Exoplanets_SolarNeighbourhood _Catalogue.csv'

# -------------------------------------------------------------------------
# 1. CARGA DE DATOS Y FUSIÓN (EU + NASA/Nuevos)
# -------------------------------------------------------------------------
print("1. Cargando y fusionando catálogos...")
df = pd.read_csv(file_csv_raw)
df['_es_nuevo'] = False

try:
    # Leemos los planetas extra (probamos con coma por defecto, o punto y coma si usaste Excel)
    try:
        df_nuevos = pd.read_csv(file_nuevos, sep=',')
        if len(df_nuevos.columns) < 2: raise ValueError
    except:
        df_nuevos = pd.read_csv(file_nuevos, sep=';')
        
    df_nuevos['_es_nuevo'] = True
    df = pd.concat([df, df_nuevos], ignore_index=True)
    print(f"   -> ¡Añadidos {len(df_nuevos)} planetas extra desde '{file_nuevos}'!")
except FileNotFoundError:
    print(f"   [!] No se encontró '{file_nuevos}'. Continuando solo con el catálogo base.")
except Exception as e:
    print(f"   [!] Error leyendo '{file_nuevos}': {e}")

cif_names_set = set()
try:
    print("   -> Cruzando con Cif25...")
    with fits.open(file_fits_cif) as hdul:
        data = hdul[1].data 
        df_cif = pd.DataFrame(np.array(data).byteswap().newbyteorder())

    for col in df_cif.columns:
        if df_cif[col].dtype == object:
            try: df_cif[col] = df_cif[col].str.decode('utf-8').str.strip()
            except: pass 
    
    mapping_cols = {'star_teff': 'Teff', 'star_mass': 'Mass_1', 'star_radius': 'Rad', 'star_sp_type': 'SpType'}
    mapping_errs = {'star_teff_error': 'e_Teff', 'star_mass_error': 'e_Mass', 'star_radius_error': 'e_Rad'}

    if 'name_2' in df_cif.columns:
        df_cif_indexed = df_cif.set_index('name_2')
        for idx_csv, row_csv in df.iterrows():
            name_p = str(row_csv['name']).strip()
            if name_p in df_cif_indexed.index:
                match = df_cif_indexed.loc[name_p]
                row_fits = match.iloc[0] if isinstance(match, pd.DataFrame) else match
                cif_names_set.add(name_p)
                
                for col_csv, col_fits in mapping_cols.items():
                    if col_fits in row_fits and pd.notna(row_fits[col_fits]):
                        df.at[idx_csv, col_csv] = row_fits[col_fits]
                
                for col_base, col_err in mapping_errs.items():
                    if col_err in row_fits and pd.notna(row_fits[col_err]):
                        df.at[idx_csv, f'{col_base}_min'] = row_fits[col_err]
                        df.at[idx_csv, f'{col_base}_max'] = row_fits[col_err]
        print(f"   -> Cruzados {len(cif_names_set)} sistemas con Cif25.")
except Exception as e:
    print(f"ERROR leyendo FITS: {e}")

# -------------------------------------------------------------------------
# 2. INICIALIZACIÓN DE LA CAPA BASE DE REFERENCIAS (NASA vs EU vs CIF)
# -------------------------------------------------------------------------
print("2. Generando trazabilidad base de referencias...")
parametros_rastreados = [
    'orbital_period', 'radius', 'mass', 'semi_major_axis', 'eccentricity', 
    'inclination', 'tzero_tr', 'tperi', 'omega', 'star_radius', 
    'star_mass', 'star_teff', 'star_distance', 'star_sp_type'
]

LINK_CIF = '<a href="https://ui.adsabs.harvard.edu/abs/2025A%26A...693A.228C/abstract" target="_blank" style="text-decoration: none; color: #007bff; font-weight:bold;">Cif25</a>'

for param in parametros_rastreados:
    col_ref = f"{param}_ref"
    if col_ref not in df.columns:
        df[col_ref] = "" 
    
    for idx, row in df.iterrows():
        p_name = str(row['name']).strip()
        is_stellar = param.startswith('star_')
        
        # SI ES UN PLANETA NUEVO (NASA)
        if row.get('_es_nuevo') == True:
            ref_texto = str(row.get('referencia_general', 'NASA')).strip()
            if ref_texto == 'nan' or ref_texto == '': ref_texto = 'NASA'
            
            # Si no le has puesto un link HTML a mano, se lo creamos en naranja
            if '<a href' not in ref_texto:
                ref_final = f'<a href="https://exoplanetarchive.ipac.caltech.edu/" target="_blank" style="text-decoration: none; color: #ff7b00; font-weight:bold;">{ref_texto}</a>'
            else:
                ref_final = ref_texto
            df.at[idx, col_ref] = ref_final
            
        # SI ES UN PLANETA DE EXOPLANET.EU
        else:
            if is_stellar and p_name in cif_names_set: 
                df.at[idx, col_ref] = LINK_CIF
            else: 
                slug = p_name.lower().replace(' ', '_').replace('+', '_').replace('-', '_')
                df.at[idx, col_ref] = f'<a href="http://exoplanet.eu/catalog/{slug}/" target="_blank" style="text-decoration: none; color: #d63384; font-weight:bold;">Eu</a>'

# -------------------------------------------------------------------------
# 3. CÁLCULO DE DISTANCIAS CINEMÁTICAS DESDE SIMBAD (GAIA/HIP)
# -------------------------------------------------------------------------
print("3. Inyectando paralajes y recalculando distancias...")
try:
    df_gaia = pd.read_csv('distancias_gaia.csv')
    gaia_dict = df_gaia.set_index('nombre_estrella').to_dict('index')
    sistemas_actualizados = 0

    for idx, row in df.iterrows():
        p_name = str(row['name']).strip()
        s_name = str(row.get('star_name', '')).strip()
        if not s_name or s_name == 'nan':
            s_name = p_name.rsplit(' ', 1)[0]
            
        if s_name in gaia_dict:
            plx = float(gaia_dict[s_name]['parallax'])
            e_plx = float(gaia_dict[s_name]['parallax_error'])
            ref_html = str(gaia_dict[s_name].get('parallax_ref', ''))
            
            if plx > 0:
                d = 1000.0 / plx                   
                e_d = d * (e_plx / plx)            
                
                df.at[idx, 'star_distance'] = d
                df.at[idx, 'star_distance_error_min'] = e_d
                df.at[idx, 'star_distance_error_max'] = e_d
                
                if ref_html and ref_html != 'nan':
                    df.at[idx, 'star_distance_ref'] = ref_html
                sistemas_actualizados += 1

    print(f"   -> ¡Distancias calculadas para {sistemas_actualizados} sistemas estelares!")
except FileNotFoundError:
    print("   [!] 'distancias_gaia.csv' no encontrado. Se usarán las distancias viejas.")

# -------------------------------------------------------------------------
# 3.2. INYECCIÓN DE REFERENCIAS DE DESCUBRIMIENTO
# -------------------------------------------------------------------------
print("3.5. Inyectando referencias de descubrimiento...")
file_discovery = 'discovery_refs.csv'
try:
    try:
        df_disc = pd.read_csv(file_discovery, sep=';', skipinitialspace=True, encoding='utf-8-sig')
        if 'name' not in [c.strip().lower() for c in df_disc.columns]: raise ValueError
    except:
        df_disc = pd.read_csv(file_discovery, sep=',', skipinitialspace=True, encoding='utf-8-sig')

    df_disc.columns = [c.strip().lower() for c in df_disc.columns]
    disc_dict = df_disc.set_index('name')['discovery_ref'].to_dict()

    df['discovery_ref'] = df['name'].map(disc_dict).fillna('')
    encontrados = df['discovery_ref'].astype(bool).sum()
    print(f"   -> Referencias de descubrimiento inyectadas: {encontrados} de {len(df)} planetas.")
except FileNotFoundError:
    df['discovery_ref'] = ''
    print(f"   [!] '{file_discovery}' no encontrado. Columna 'discovery_ref' inicializada vacía.")
except Exception as e:
    df['discovery_ref'] = ''
    print(f"   [!] Error procesando discovery_refs: {e}")
    

# -------------------------------------------------------------------------
# 4. APLICACIÓN DEL CATÁLOGO DE PARCHES MANUALES
# -------------------------------------------------------------------------
print("4. Aplicando correcciones manuales de la literatura...")
try:
    try:
        df_parches = pd.read_csv(file_correcciones, sep=';', skipinitialspace=True, encoding='utf-8-sig')
        if 'name' not in [c.strip().lower() for c in df_parches.columns]: raise ValueError
    except:
        df_parches = pd.read_csv(file_correcciones, sep=',', skipinitialspace=True, encoding='utf-8-sig')
        
    df_parches.columns = [c.strip().lower() for c in df_parches.columns]
    
    modificados, borrados = 0, 0
    
    for _, row_parche in df_parches.iterrows():
        if pd.isna(row_parche.get('name')) or pd.isna(row_parche.get('accion')): continue
            
        # MAGIA: Separamos los nombres por la barra vertical (|)
        nombres_planetas = [n.strip() for n in str(row_parche['name']).split('|')]
        accion = str(row_parche['accion']).strip().lower()
        
        if accion == 'borrar':
            for p_name in nombres_planetas:
                df = df[df['name'] != p_name]
                borrados += 1
                print(f"   [-] Eliminado: {p_name}")
            continue
            
        if accion == 'modificar':
            param = str(row_parche.get('parametro', '')).strip()
            if not param or str(param).strip() == '': continue
            
            n_val = row_parche.get('nuevo_valor', np.nan)
            n_err_min = row_parche.get('nuevo_error_min', np.nan)
            n_err_max = row_parche.get('nuevo_error_max', np.nan)
            n_ref = row_parche.get('nueva_ref', np.nan)

            # BUCLE: Aplicamos el cambio a cada planeta de la lista
            for p_name in nombres_planetas:
                idx_list = df[df['name'] == p_name].index
                if len(idx_list) == 0: continue
                idx_df = idx_list[0]
                
                # Función interna adaptada para recibir el índice (idx_df)
                def aplicar_o_vaciar(columna, valor, indice):
                    if pd.isna(valor) or str(valor).strip() == '': return
                    if columna not in df.columns: df[columna] = np.nan
                        
                    if str(valor).strip().upper() == 'BORRAR':
                        df.at[indice, columna] = np.nan
                    else:
                        try:
                            df.at[indice, columna] = float(valor)
                        except ValueError:
                            df.at[indice, columna] = valor

                aplicar_o_vaciar(param, n_val, idx_df)
                aplicar_o_vaciar(f"{param}_error_min", n_err_min, idx_df)
                aplicar_o_vaciar(f"{param}_error_max", n_err_max, idx_df)
                aplicar_o_vaciar(f"{param}_ref", n_ref, idx_df)
                
                modificados += 1

    df.reset_index(drop=True, inplace=True) 
    print(f"   -> Resumen: {borrados} borrados | {modificados} parámetros editados.")
except Exception as e:
    print(f"   [!] Error procesando parches: {e}")

# -------------------------------------------------------------------------
# 5. RE-CÁLCULOS FÍSICOS FINALES Y EXPORTACIÓN
# -------------------------------------------------------------------------
print("5. Recalculando parámetros derivados y exportando...")

# --- NUEVO: ESTANDARIZAR EL STATUS DEL PLANETA ---
if 'planet_status' not in df.columns:
    df['planet_status'] = 'Confirmed'
df['planet_status'] = df['planet_status'].fillna('Confirmed').str.title()
# -------------------------------------------------

df['detection_type'] = df['detection_type'].fillna('Unknown')
df['mass_measurement_type'] = df['mass_measurement_type'].fillna('Unknown')
df['radius_measurement_type'] = df['radius_measurement_type'].fillna('Unknown')
df['star_sp_type'] = df['star_sp_type'].fillna('')

def classify_method(row):
    det = str(row['detection_type']).lower()
    mass = str(row['mass_measurement_type']).lower()
    has_rv = 'radial velocity' in det or 'rv' in det or 'radial velocity' in mass
    has_transit = 'transit' in det and 'timing' not in det and 'ttv' not in det
    has_astro = 'astrometry' in det or 'astrometry' in mass
    has_ttv = 'ttv' in det or 'timing' in det or 'ttv' in mass or 'timing' in mass
    
    if has_ttv: return 'TTV'
    elif has_transit and has_rv: return 'Transit + RV'
    elif has_rv and has_astro: return 'RV + Astrometry'
    elif has_rv and not has_transit and not has_astro: return 'RV'
    elif has_astro and not has_rv and not has_transit: return 'Astrometry'
    elif has_transit and not has_rv: return 'Transit' 
    else: return 'Other'

df['Detection_Method_Full'] = df.apply(classify_method, axis=1)

M, R = df['star_mass'], df['star_radius']

# Inicializar la columna si no existe (puede haberla creado el paso 4 con valor manual)
if 'star_density' not in df.columns:
    df['star_density'] = np.nan
    
# ρ_☉ = M_☉ / (4/3 π R_☉³) → g/cm³  (~1.408 g/cm³)
RHO_SUN = (const.M_sun / (4/3 * np.pi * const.R_sun**3)).to(u.g / u.cm**3).value

density_calc = (M / R**3) * RHO_SUN
df['star_density'] = df['star_density'].where(df['star_density'].notna(), density_calc)

rel_M_up = df['star_mass_error_max'].abs().fillna(0) / M
rel_M_lo = df['star_mass_error_min'].abs().fillna(0) / M
rel_R_up = df['star_radius_error_max'].abs().fillna(0) / R
rel_R_lo = df['star_radius_error_min'].abs().fillna(0) / R

err_max_calc = df['star_density'] * np.sqrt( rel_M_up**2 + (3 * rel_R_lo)**2 )
err_min_calc = df['star_density'] * np.sqrt( rel_M_lo**2 + (3 * rel_R_up)**2 )

if 'star_density_error_max' not in df.columns:
    df['star_density_error_max'] = np.nan
if 'star_density_error_min' not in df.columns:
    df['star_density_error_min'] = np.nan

df['star_density_error_max'] = df['star_density_error_max'].where(df['star_density_error_max'].notna(), err_max_calc)
df['star_density_error_min'] = df['star_density_error_min'].where(df['star_density_error_min'].notna(), err_min_calc)

if 'star_density_ref' not in df.columns:
    df['star_density_ref'] = np.nan

df['star_density_ref'] = df['star_density_ref'].where(df['star_density_ref'].notna(), '-')

df['T0_val'] = df['tzero_tr'].fillna(df['tperi'])
df['T0_min'] = df['tzero_tr_error_min'].fillna(df['tperi_error_min'])
df['T0_max'] = df['tzero_tr_error_max'].fillna(df['tperi_error_max'])
df['T0_val_ref'] = df['tzero_tr_ref'].where(df['tzero_tr'].notna(), df['tperi_ref'])

# Limpiamos la columna auxiliar
df = df.drop(columns=['_es_nuevo'])

df.to_csv(file_output, index=False)
print(f"¡Todo listo! Catálogo científico maestro guardado en: {file_output}")

1. Cargando y fusionando catálogos...
   -> ¡Añadidos 11 planetas extra desde 'nuevos_planetas.csv'!
   -> Cruzando con Cif25...
   -> Cruzados 78 sistemas con Cif25.
2. Generando trazabilidad base de referencias...
3. Inyectando paralajes y recalculando distancias...
   -> ¡Distancias calculadas para 132 sistemas estelares!
3.5. Inyectando referencias de descubrimiento...
   -> Referencias de descubrimiento inyectadas: 129 de 132 planetas.
4. Aplicando correcciones manuales de la literatura...
   [-] Eliminado: Wolf 359 c
   [-] Eliminado: HD 219134 e
   [-] Eliminado: GJ 680 b
   -> Resumen: 3 borrados | 1334 parámetros editados.
5. Recalculando parámetros derivados y exportando...
¡Todo listo! Catálogo científico maestro guardado en: Exoplanets_SolarNeighbourhood _Catalogue.csv
